# Silver Layer Inventory & OBT Readiness Analysis

Notebook phân tích **hiện trạng tầng Silver** trên MinIO — kiểm kê 38 bảng, sample schema từng nhóm, đánh giá khả năng join để chuẩn bị build bảng **`obt_match_360`** (One Big Table) tầng Gold.

**Mục lục**
1. Kết nối MinIO & Kiểm kê tổng quan
2. Nhóm Matches (p03, p09, p24)
3. Nhóm Odds & Market (p03, p22)
4. Nhóm xG & Shot Data (p19/Understat)
5. Nhóm Cầu thủ (p24, p19, p08)
6. Nhóm Dim (Sân, Đội, Wikidata, TheSportsDB)
7. Nhóm Text & Media (p20, p21, p10)
8. Nhóm Thời tiết & Chấn thương (p13, p16)
9. Đánh giá khả năng join → OBT readiness matrix

## 1. Kết nối MinIO & Kiểm kê tổng quan

In [ ]:
# [DÀNH RIÊNG CHO GOOGLE COLAB]
import sys, os
os.environ['MINIO_ENDPOINT'] = 'http://20.41.113.183:9000'
os.environ['MINIO_ACCESS_KEY'] = 'minioadmin'
os.environ['MINIO_SECRET_KEY'] = 'minioadmin123'
if 'SEED_PATH' in os.environ: del os.environ['SEED_PATH']
if 'google.colab' in sys.modules:
    !rm -rf /content/Lab
    !git clone -b ml https://github.com/nbngoc123/Lab.git /content/Lab
    os.chdir('/content/Lab/football-lake')
    sys.path.insert(0, '/content/Lab/football-lake')
    !pip install minio duckdb pandas pyarrow boto3 python-dotenv -q
    print('Setup xong Colab!')


In [ ]:
import io, warnings
import numpy as np
import pandas as pd
import duckdb
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='deep')
plt.rcParams['figure.dpi'] = 110
pd.set_option('display.max_columns', 60)
pd.set_option('display.max_colwidth', 40)

from lake import minio_io as mio

print('✓ Kết nối MinIO OK')
print(f'  Endpoint: {os.environ["MINIO_ENDPOINT"]}')


In [ ]:
# ── Kiểm kê toàn bộ Silver layer ──────────────────────────────────────────────
rows = []
for key, size in mio.list_keys('silver/'):
    parts = key.split('/')
    if len(parts) >= 3:
        rows.append({
            'table_path': '/'.join(parts[:3]),
            'group': parts[1],
            'table': parts[2],
            'key': key,
            'size_bytes': size,
            'is_parquet': key.endswith('.parquet')
        })

inv = pd.DataFrame(rows)
summary = (inv[inv['is_parquet']]
    .groupby('table_path')
    .agg(n_files=('key', 'count'),
         total_size_kb=('size_bytes', lambda x: x.sum() / 1024))
    .reset_index()
    .sort_values('table_path'))

print(f'Silver tables: {len(summary)}')
print(f'Total size: {summary["total_size_kb"].sum():.0f} KB '
      f'({summary["total_size_kb"].sum()/1024:.1f} MB)')
display(summary.style.format({'total_size_kb': '{:.0f}'}))


In [ ]:
# ── Biểu đồ: size theo nhóm ───────────────────────────────────────────────────
summary['group'] = summary['table_path'].str.split('/').str[1]
group_size = summary.groupby('group')['total_size_kb'].sum().sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(9, 5))
colors = sns.color_palette('viridis', len(group_size))
group_size.plot(kind='barh', ax=ax, color=colors, edgecolor='white')
ax.set_title('Dung lượng Silver theo nhóm (KB)', fontweight='bold')
ax.set_xlabel('KB')
for i, (idx, val) in enumerate(group_size.items()):
    ax.text(val + 20, i, f'{val:.0f} KB', va='center', fontsize=9)
plt.tight_layout(); plt.show()


## Helper: đọc bảng Silver từ MinIO

In [ ]:
def read_silver(prefix, max_files=10):
    """Đọc tất cả parquet trong prefix, trả về DataFrame."""    dfs = []
    for key, _ in mio.list_keys(prefix + '/'):
        if key.endswith('.parquet'):
            try:
                raw = mio.read_bytes(key)
                dfs.append(pd.read_parquet(io.BytesIO(raw)))
                if len(dfs) >= max_files:
                    break
            except Exception as e:
                print(f'  ! {key}: {e}')
    if not dfs:
        print(f'  ! Không có data: {prefix}')
        return pd.DataFrame()
    df = pd.concat(dfs, ignore_index=True)
    print(f'[{prefix.split("/")[-1]}] {len(df):,} dòng × {len(df.columns)} cột')
    return df

def schema_summary(df, name=''):
    """In schema + null % + sample."""    if df.empty: return
    null_pct = (df.isna().mean() * 100).round(1)
    schema = pd.DataFrame({
        'dtype': df.dtypes,
        'null_%': null_pct,
        'sample': df.iloc[0] if len(df) else None
    })
    print(f'\n{name} — {len(df):,} rows')
    display(schema)


## 2. Nhóm Matches

Các bảng trận đấu là **spine** (xương sống) của OBT — mọi bảng khác sẽ join vào đây.

| Bảng | Nguồn | Mô tả |
|------|-------|-------|
| `fd_matches` | p03 football-data.co.uk | Lịch sử kết quả + odds thô (2020–2026) |
| `af_fixtures` | p24 API-Football | Fixtures EPL 2024 + score + status |
| `af_match_stats` | p24 API-Football | Thống kê sau trận (possession, shots...) |
| `af_match_events` | p24 API-Football | Sự kiện trong trận (goal, card, sub) |
| `af_lineups` | p24 API-Football | Đội hình ra sân |
| `understat_match_xg` | p19 Understat | xG / xGA per trận |


In [ ]:
fd = read_silver('silver/matches/fd_matches')
schema_summary(fd, 'fd_matches')
print('\nSeasons:', sorted(fd['season'].unique()) if 'season' in fd.columns else 'N/A')
print('Date range:', fd['match_date'].min(), '→', fd['match_date'].max() if 'match_date' in fd.columns else 'N/A')


In [ ]:
af_fx = read_silver('silver/matches/af_fixtures')
schema_summary(af_fx, 'af_fixtures')


In [ ]:
af_stats = read_silver('silver/matches/af_match_stats')
schema_summary(af_stats, 'af_match_stats')


In [ ]:
af_events = read_silver('silver/matches/af_match_events')
print('Event types:', af_events['type'].value_counts().head(10).to_dict() if 'type' in af_events.columns else 'N/A')
schema_summary(af_events, 'af_match_events')


In [ ]:
af_lineups = read_silver('silver/matches/af_lineups')
schema_summary(af_lineups, 'af_lineups')


In [ ]:
u_match = read_silver('silver/matches/understat_match_xg')
schema_summary(u_match, 'understat_match_xg')
print('Leagues:', u_match['league'].unique() if 'league' in u_match.columns else 'N/A')


## 3. Nhóm Odds & Market

| Bảng | Nguồn | Mô tả |
|------|-------|-------|
| `fd_odds` | p03 | Odds 1X2 từ nhiều nhà cái (lịch sử) |
| `odds_h2h` | p22 The Odds API | Kèo 1X2 upcoming |
| `odds_spreads` | p22 | Kèo Handicap Asian |
| `odds_totals` | p22 | Kèo Tài/Xỉu |


In [ ]:
fd_odds = read_silver('silver/odds/fd_odds')
schema_summary(fd_odds, 'fd_odds')
print('Bookmakers:', fd_odds['bookmaker'].unique() if 'bookmaker' in fd_odds.columns else 'N/A')


In [ ]:
odds_h2h = read_silver('silver/betting/odds_h2h')
schema_summary(odds_h2h, 'odds_h2h (upcoming)')


## 4. Nhóm xG & Shot Data (Understat)

| Bảng | Mô tả |
|------|-------|
| `understat_team_xg` | xG / xGA / PPDA per đội per trận |
| `understat_player_xg` | Thống kê xG cầu thủ (goals, assists, xG, xA) |
| `understat_shots` | Tọa độ từng cú sút (6 giải × 2 mùa) |


In [ ]:
u_team = read_silver('silver/teams/understat_team_xg')
schema_summary(u_team, 'understat_team_xg')
print('Leagues:', u_team['league'].unique() if 'league' in u_team.columns else 'N/A')
print('Seasons:', u_team['season'].unique() if 'season' in u_team.columns else 'N/A')


In [ ]:
u_player = read_silver('silver/players/understat_player_xg')
schema_summary(u_player, 'understat_player_xg')


In [ ]:
u_shots = read_silver('silver/events/understat_shots', max_files=3)
schema_summary(u_shots, 'understat_shots (sample 3 files)')
print('Total shot files on MinIO: see inventory above')


## 5. Nhóm Cầu thủ

| Bảng | Nguồn | Mô tả |
|------|-------|-------|
| `af_players` | p24 API-Football | Profile + season stats cầu thủ EPL |
| `af_top_scorers` | p24 | Top ghi bàn EPL 2024 |
| `af_injuries` | p24 | Lịch sử chấn thương 2024 (3168 bản ghi) |
| `tsdb_players` | p08 TheSportsDB | Profile cầu thủ (ảnh, DOB, quốc tịch) |


In [ ]:
af_players = read_silver('silver/players/af_players')
schema_summary(af_players, 'af_players')


In [ ]:
af_inj = read_silver('silver/players/af_injuries')
schema_summary(af_inj, 'af_injuries')
print('Injury reasons:', af_inj['reason'].value_counts().head(8).to_dict() if 'reason' in af_inj.columns else 'N/A')


In [ ]:
tsdb_players = read_silver('silver/dim/tsdb_players')
schema_summary(tsdb_players, 'tsdb_players')


## 6. Nhóm Dimension Tables

| Bảng | Nguồn | Mô tả |
|------|-------|-------|
| `wd_stadiums` | p06 Wikidata | Sân vận động (tọa độ, sức chứa) |
| `wd_clubs` | p06 | CLB (thành lập, màu áo) |
| `tsdb_teams` | p08 TheSportsDB | Logo, tên đầy đủ, quốc gia |
| `af_standings` | p24 | Bảng xếp hạng EPL |


In [ ]:
wd_stadiums = read_silver('silver/dim/wd_stadiums')
schema_summary(wd_stadiums, 'wd_stadiums')

tsdb_teams = read_silver('silver/dim/tsdb_teams')
schema_summary(tsdb_teams, 'tsdb_teams')

af_standings = read_silver('silver/standings/af_standings')
schema_summary(af_standings, 'af_standings')


## 7. Nhóm Text & Media

| Bảng | Nguồn | Mô tả |
|------|-------|-------|
| `google_news_articles` | p20 | Bài báo Google News theo đội |
| `wm_pageviews` | p10 Wikimedia | Lượt xem Wikipedia theo ngày |
| `youtube_videos` | p21 | Video highlights |
| `youtube_comments` | p21 | Bình luận fans |
| `football_news_agg` | p25 | Tin tức tổng hợp |


In [ ]:
news = read_silver('silver/text/google_news_articles')
schema_summary(news, 'google_news_articles')
print('Date range:', news['published_ts'].min(), '→', news['published_ts'].max() if 'published_ts' in news.columns else 'N/A')


In [ ]:
pv = read_silver('silver/text/wm_pageviews')
schema_summary(pv, 'wm_pageviews')
print('Date range:', pv['date'].min(), '→', pv['date'].max() if 'date' in pv.columns else 'N/A')
print('Entities:', pv['entity'].unique() if 'entity' in pv.columns else 'N/A')


In [ ]:
yt_videos = read_silver('silver/text/youtube_videos')
yt_comments = read_silver('silver/text/youtube_comments')
schema_summary(yt_videos, 'youtube_videos')
schema_summary(yt_comments, 'youtube_comments (sample)')


## 8. Nhóm Thời tiết & Chấn thương

| Bảng | Nguồn | Mô tả |
|------|-------|-------|
| `fd_matches_weather` | p13 Open-Meteo | Thời tiết per trận per sân |
| `om_venue_weather` | p13 | Hourly weather per venue |
| `pr_player_injuries` | p16 PhysioRoom | Snapshot chấn thương hiện tại |
| `pr_club_injury_summary` | p16 | Tổng hợp chấn thương theo CLB |


In [ ]:
weather = read_silver('silver/matches/fd_matches_weather')
schema_summary(weather, 'fd_matches_weather')
print('Matches with weather:', weather['match_id'].nunique() if 'match_id' in weather.columns else 'N/A')

om_venue = read_silver('silver/dim/om_venue_weather')
print(f'om_venue_weather: {len(om_venue):,} hourly records')
print('Venues:', om_venue['venue'].nunique() if 'venue' in om_venue.columns else 'N/A')


In [ ]:
pr_inj = read_silver('silver/players/pr_player_injuries')
pr_club = read_silver('silver/dim/pr_club_injury_summary')
schema_summary(pr_inj, 'pr_player_injuries')
schema_summary(pr_club, 'pr_club_injury_summary')


## 9. OBT Readiness Matrix

Đánh giá khả năng join của từng bảng Silver vào bảng `obt_match_360` (grain: 1 dòng/trận).

**Join key chính:**
- `match_id` (format `E0_YYYYMMDD_HomeAway` từ p03)  ← spine
- `fixture_id` (API-Football) ← bridge qua `bridge_fd_af_match`
- `(home_key, away_key, match_date)` ← fuzzy join dự phòng


In [ ]:
# ── Đánh giá readiness ────────────────────────────────────────────────────────
readiness = [
    # (Bảng, Join Key, Tình trạng, Ghi chú)
    ('fd_matches',           'match_id',                   '✅ Ready',   'Spine — 760 trận 2024-26'),
    ('fd_odds',              'match_id',                   '✅ Ready',   'Odds 1X2 lịch sử đầy đủ'),
    ('fd_matches_weather',   'match_id',                   '⚠️ Partial', 'p13 chưa phủ đủ sân (429 error)'),
    ('af_fixtures',          'bridge fixture_id',          '✅ Ready',   '380 trận EPL 2024'),
    ('af_match_stats',       'bridge fixture_id',          '⚠️ Partial', 'Mới có 72/380 trận (p24 đang cào)'),
    ('af_match_events',      'bridge fixture_id',          '⚠️ Partial', 'Mới có 72/380 trận'),
    ('af_lineups',           'bridge fixture_id',          '⚠️ Partial', 'Mới có 72/380 trận'),
    ('understat_team_xg',    '(team_key, match_date)',     '✅ Ready',   '6 giải × 2 mùa (2024+2025)'),
    ('understat_match_xg',   '(team_key, match_date)',     '✅ Ready',   'xG per trận'),
    ('understat_shots',      '(league, match_id)',         '✅ Ready',   'Shot coords 6 giải'),
    ('af_players',           'player_id',                  '✅ Ready',   '40 players profile'),
    ('af_injuries',          '(team_key, season)',         '✅ Ready',   '3168 ca chấn thương EPL 2024'),
    ('understat_player_xg',  '(player_name, league)',      '✅ Ready',   'xG stats cầu thủ'),
    ('wd_stadiums',          'team_key (venue)',           '✅ Ready',   'Tọa độ sân, sức chứa'),
    ('tsdb_teams',           'team_key',                   '✅ Ready',   'Logo, metadata đội'),
    ('af_standings',         '(team_key, season)',         '✅ Ready',   'BXH cuối mùa'),
    ('odds_h2h / spreads',   '(home_key,away_key,date)',   '⚠️ Future',  'Chỉ có upcoming, không có lịch sử'),
    ('google_news_articles', '(team_key, date window)',    '⚠️ Limited', 'Cần rolling aggregation'),
    ('wm_pageviews',         '(team_key, date)',           '✅ Ready',   '67K dòng 2023–2026'),
    ('youtube_comments',     '(query, date)',              '⚠️ Limited', 'Chưa join được trực tiếp'),
    ('pr_player_injuries',   '(team_key, ASOF date)',      '⚠️ Partial', 'Chỉ 3 snapshots'),
]

df_ready = pd.DataFrame(readiness, columns=['Silver Table', 'Join Key', 'Status', 'Ghi chú'])
display(df_ready.style.apply(
    lambda col: ['background: #d4edda' if '✅' in v else
                 'background: #fff3cd' if '⚠️' in v else ''
                 for v in col] if col.name == 'Status' else ['' for _ in col],
    axis=0
))

# Summary
n_ready = sum('✅' in r[2] for r in readiness)
n_partial = sum('⚠️' in r[2] for r in readiness)
print(f'\n✅ Ready: {n_ready} bảng | ⚠️ Partial/Future: {n_partial} bảng')


## 10. Schema đề xuất cho `obt_match_360`

Grain: **1 dòng = 1 trận đấu (home perspective)**

```
obt_match_360
├── KEY: match_id, fixture_id, division, season, match_date
├── TEAMS: home_team, away_team, home_key, away_key
├── RESULT: home_goals, away_goals, result, target_over25, target_btts
│
├── FORM (từ fd_matches rolling):
│   home_pts_l5, home_gf_l5, home_ga_l5, away_pts_l5...
│
├── MARKET (từ fd_odds):
│   mkt_p_home, mkt_p_draw, mkt_p_away, pin_p_home...
│
├── xG (từ understat_team_xg):
│   home_xg_l5, home_xga_l5, home_xgd_l5, away_xg_l5...
│
├── STATS_AF (từ af_match_stats via bridge):
│   home_poss_l5, home_sog_l5, home_shots_l5...
│
├── MATCH_EVENTS (từ af_match_events):
│   home_goals_first_half, home_red_cards, home_subs...
│
├── LINEUP (từ af_lineups):
│   home_avg_age, home_foreign_count, home_formation...
│
├── INJURIES (từ af_injuries / pr_player_injuries ASOF):
│   home_injured_count, away_injured_count...
│
├── WEATHER (từ fd_matches_weather):
│   temp_c, precip_mm, wind_kmh, is_raining
│
├── ATTENTION (từ wm_pageviews rolling):
│   home_pv_7d, home_pv_28d, home_pv_ratio_7_28...
│
└── DIM: venue, city, capacity (wd_stadiums)
         home_logo, away_logo (tsdb_teams)
```


In [ ]:
# ── Ước tính độ phủ từng nhóm column trong OBT ────────────────────────────────
# Dựa trên hiện trạng Silver vừa kiểm kê

groups = {
    'form':        ('fd_matches',     760, 760,  '✅ 100% — spine'),
    'market':      ('fd_odds',        760, 740,  '✅ 97% — hầu hết trận có odds'),
    'xG':          ('understat_xg',   760, 760,  '✅ 100% — 2 mùa đã cào'),
    'stats_af':    ('af_match_stats', 760,  72,  '⚠️ 9.5% — cần ~15 ngày nữa'),
    'events_af':   ('af_events',      760,  72,  '⚠️ 9.5% — cùng batch với stats'),
    'lineups':     ('af_lineups',     760,  72,  '⚠️ 9.5% — cùng batch'),
    'injuries_af': ('af_injuries',    760, 760,  '✅ 100% — 3168 ca cả mùa'),
    'weather':     ('fd_weather',     760, 304,  '⚠️ 40% — p13 bị 429, cần chạy lại'),
    'attention':   ('wm_pageviews',   760, 708,  '✅ 93% — 67K dòng 2023-2026'),
    'injuries_pr': ('physioroom',     760,  14,  '⚠️ 2% — chỉ 3 snapshots'),
}

rows = []
for grp, (src, total, covered, note) in groups.items():
    rows.append({'Group': grp, 'Source': src, 'Total': total,
                 'Covered': covered, 'Coverage': covered/total, 'Note': note})

df_cov = pd.DataFrame(rows).sort_values('Coverage', ascending=True)

fig, ax = plt.subplots(figsize=(10, 5))
colors = ['#5cb85c' if v >= 0.8 else '#f0ad4e' if v >= 0.3 else '#d9534f'
          for v in df_cov['Coverage']]
ax.barh(df_cov['Group'], df_cov['Coverage'], color=colors, edgecolor='white')
ax.axvline(1.0, ls='--', color='green', alpha=0.4)
ax.axvline(0.5, ls='--', color='orange', alpha=0.4)
ax.set_xlim(0, 1.1)
ax.set_xlabel('Coverage (% trận có data)')
ax.set_title('OBT Column Group Coverage — Hiện trạng', fontweight='bold')
for i, (_, row) in enumerate(df_cov.iterrows()):
    ax.text(row['Coverage'] + 0.01, i, f'{row["Coverage"]:.0%}  {row["Note"]}',
            va='center', fontsize=8)
plt.tight_layout(); plt.show()


## 11. Action Plan → Build `obt_match_360`

| Bước | Action | Priority |
|------|--------|----------|
| 1 | Chạy lại `p13` (thời tiết) — đã fix 429 handler | 🔴 Cao |
| 2 | Tiếp tục `p24` hàng ngày để cào events/lineups/stats | 🟡 Trung bình |
| 3 | Chạy `build_obt_match.py` join các bảng ✅ Ready | 🔴 Cao |
| 4 | Thêm cột events (goals 1H/2H, red cards) từ af_events | 🟡 Sau khi p24 đủ |
| 5 | Thêm cột lineup (formation, avg_age) từ af_lineups | 🟡 Sau khi p24 đủ |
| 6 | Thêm cột xG per match từ understat_shots (agg) | 🟢 Có thể làm ngay |

**Lệnh chạy build OBT (sau khi viết xong `build_obt_match.py`):**
```bash
cd ~/lab/Lab/football-lake
git pull origin ml
source ~/venv/bin/activate
export PYTHONPATH=/home/azureuser/lab/Lab/football-lake
python notebooks/python/build_obt_match.py
```
